In [1]:
import json
import numpy as np
import pandas as pd
from io import StringIO
import textwrap
from model_inference.gpt import *
from utils.table_utils import *

# Table parsing test

In [2]:
path = '../data/livesum/test.json'

In [3]:
df = pd.read_json(path)

In [4]:
print(df.head())

                                                text  \
0  And we're off for the first half. Player27(Awa...   
1  The game is underway with the start of the fir...   
2  The game is underway with the start of the fir...   
3  And we're off for the first half. Player26(Awa...   
4  The game is underway with the start of the fir...   

                                               table        id  
0  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25513332  
1  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25513360  
2  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25600389  
3  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25617902  
4  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25892175  


In [5]:
print(df.columns)

Index(['text', 'table', 'id'], dtype='object')


In [6]:
idx = 3

In [7]:
print(df['table'][idx])

Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,Corner Kicks,Free Kicks,Offsides<NEWLINE>Away Team,2,8,9,2,0,5,12,5<NEWLINE>Home Team,1,11,12,2,0,3,9,4


In [8]:
table_string = df['table'][idx]
table_string = table_string.replace('<NEWLINE>', '\n')

In [9]:
table_string_io = StringIO(table_string)

In [10]:
df_table = pd.read_csv(table_string_io)

In [11]:
print(df_table.to_string(index=False))

     Team  Goals  Shots  Fouls  Yellow Cards  Red Cards  Corner Kicks  Free Kicks  Offsides
Away Team      2      8      9             2          0             5          12         5
Home Team      1     11     12             2          0             3           9         4


# Prompting test

In [12]:
df = pd.read_json('../data/livesum/test.json')
%clear
print(textwrap.fill(df['text'][idx], width=100))

And we're off for the first half. Player26(Away Team) earns a free kick in their defensive half
following a foul by Player7(Home Team). Player7(Home Team) receives a yellow card for a rough
tackle. The Away Team earns a corner kick. Player27(Away Team)'s shot from the center of the box,
assisted by Player20(Away Team), just missed to the left. Player10(Home Team)'s move is risky.
Player22(Away Team) earns a free kick in their own half. Player27(Away Team)'s left footed shot from
outside the box was blocked with an assist from Player28(Away Team). Player6(Home Team) earns a free
kick in their own half. Player27(Away Team) commits a foul. Player8(Home Team)'s left footed shot
from outside the box goes wide to the right, with an assist from Player9(Home Team). Player25(Away
Team)'s left footed shot from outside the box, assisted by Player28(Away Team), is blocked after the
Away Team's corner kick is obtained. Player5(Home Team) is currently sidelined due to an injury,
causing a delay in t

In [13]:
text = df['text'][idx]
atomic_out = ask_chatgpt(text=text,prompt_path="prompts/livesum_atomic.txt")
print(atomic_out)

Player26 (Away Team) earns a free kick in their defensive half following a foul by Player7 (Home Team).  
Player7 (Home Team) receives a yellow card for a rough tackle.  
The Away Team earns a corner kick.  
Player27 (Away Team) takes a shot from the center of the box.  
Player20 (Away Team) assists Player27 (Away Team).  
Player27 (Away Team)'s shot misses to the left.  
Player10 (Home Team) makes a risky move.  
Player22 (Away Team) earns a free kick in their own half.  
Player27 (Away Team) takes a left-footed shot from outside the box.  
Player28 (Away Team) assists Player27 (Away Team).  
Player27 (Away Team)'s shot is blocked.  
Player6 (Home Team) earns a free kick in their own half.  
Player27 (Away Team) commits a foul.  
Player8 (Home Team) takes a left-footed shot from outside the box.  
Player9 (Home Team) assists Player8 (Home Team).  
Player8 (Home Team)'s shot goes wide to the right.  
Player25 (Away Team) takes a left-footed shot from outside the box.  
Player28 (Away T

In [14]:
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'w') as f:
    f.write(atomic_out)

In [15]:
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'r') as f:
    atomic_text = f.read()
header_out = ask_chatgpt(text=atomic_text,prompt_path="prompts/livesum_header.txt")
print(header_out)

{
  "row_headers": ["Player1 (Home Team)", "Player2 (Home Team)", "Player4 (Home Team)", "Player5 (Home Team)", "Player6 (Home Team)", "Player7 (Home Team)", "Player8 (Home Team)", "Player9 (Home Team)", "Player10 (Home Team)", "Player11 (Home Team)", "Player12 (Home Team)", "Player20 (Away Team)", "Player21 (Away Team)", "Player22 (Away Team)", "Player23 (Away Team)", "Player24 (Away Team)", "Player25 (Away Team)", "Player26 (Away Team)", "Player27 (Away Team)", "Player28 (Away Team)", "Player29 (Away Team)", "Player30 (Away Team)", "Player32 (Away Team)", "Player35 (Away Team)"],
  "column_headers": ["Goals", "Assists", "Yellow Cards", "Fouls Committed", "Free Kicks Earned", "Shots Taken", "Shots Saved", "Shots Blocked", "Offsides", "Injuries", "Corner Kicks"]
}


In [16]:
with open('./model_outputs/gpt_livesum_test/header_each.txt', 'w') as f:
    f.write(header_out)

In [27]:
with open('./model_outputs/gpt_livesum_test/header_each.txt', 'r') as f:
    header_text = f.read()
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'r') as f:
    atomic_text = f.read()
input_text = header_text + '\n' + atomic_text
output_table = ask_chatgpt(text=input_text,prompt_path="prompts/livesum_table_new.txt")
print(output_table)

|  | Goals | Assists | Yellow Cards | Fouls Committed | Free Kicks Earned | Shots Taken | Shots Saved | Shots Blocked | Offsides | Injuries | Corner Kicks |
| Player1 (Home Team) | Not found | Not found | Not found | Not found | Not found | Not found | Not found | Not found | Not found | Not found | Not found |
| Player2 (Home Team) | Not found | 3 | Not found | Not found | 2 | Not found | Not found | Not found | Not found | Not found | 2 |
| Player4 (Home Team) | Not found | Not found | Not found | Not found | 1 | Not found | Not found | Not found | 1 | Not found | Not found |
| Player5 (Home Team) | 1 | Not found | Not found | 1 | Not found | Not found | Not found | Not found | Not found | 1 | Not found |
| Player6 (Home Team) | Not found | Not found | Not found | Not found | 1 | Not found | Not found | Not found | Not found | Not found | Not found |
| Player7 (Home Team) | 1 | 1 | 1 | 3 | 1 | 3 | Not found | 1 | 2 | Not found | Not found |
| Player8 (Home Team) | 1 | 1 | 1 | 1 | 1 |

In [28]:
convert_to_df(output_table)

,,Goals,Assists,Yellow Cards,Fouls Committed,Free Kicks Earned,Shots Taken,Shots Saved,Shots Blocked,Offsides,Injuries,Corner Kicks
0,Player1 (Home Team),Not found,Not found,Not found,Not found,Not found,Not found,Not found,Not found,Not found,Not found,Not found
1,Player2 (Home Team),Not found,3,Not found,Not found,2,Not found,Not found,Not found,Not found,Not found,2
2,Player4 (Home Team),Not found,Not found,Not found,Not found,1,Not found,Not found,Not found,1,Not found,Not found
3,Player5 (Home Team),1,Not found,Not found,1,Not found,Not found,Not found,Not found,Not found,1,Not found
4,Player6 (Home Team),Not found,Not found,Not found,Not found,1,Not found,Not found,Not found,Not found,Not found,Not found
5,Player7 (Home Team),1,1,1,3,1,3,Not found,1,2,Not found,Not found
6,Player8 (Home Team),1,1,1,1,1,2,1,Not found,1,Not found,2
7,Player9 (Home Team),1,2,3,3,1,3,1,Not found,1,Not found,Not found
8,Player10 (Home Team),Not found,Not found,1,1,1,1,Not found,Not found,1,Not found,Not found
9,Player11 (Home Team),Not found,Not found,Not found,3,1,1,1,Not found,Not found,1,Not found


In [29]:
print(df_table.to_string(index=False))

     Team  Goals  Shots  Fouls  Yellow Cards  Red Cards  Corner Kicks  Free Kicks  Offsides
Away Team      2      8      9             2          0             5          12         5
Home Team      1     11     12             2          0             3           9         4
